# Training and testing

This notebook focuses on training our ML algorithm to screen whether a player is at risk of gambling-related behavioral problems. It will do so by observing the players' actions while gambling. These actions are then fed into an XGBoost ML algorithm trained on real gambling data from our Bustabit dataset. The features that the game will record are behaviors that scientists use to determine at risk gambling patterns through gameplay. These actions are then fed to our XGBoost at risk gambling screening model. 

## The steps:

1. First, we will create a table that has all of the “players_stats” that will have each row be a summerization of their betting behaviors using measurable features (loss chasing, average bet size, etc...)
2. Create a column for every player that shows whether a player is at risk of being addicted to gambling (low, medium, or high).
3. Create an SMOTE class-balancing step, since most players are low risk based on the data visualization we did in previous notebooks.
4. Training and Testing using Stratified 5-fold cross-validation [K-fold cross-validation]
5. Train the XGBoost model on the training data after K-fold to demonstrate that the approach actually works.
6. Measure how well the model performs using charts and other metrics.

### Important!
    This is just a screening tool to help people who might be at risk of a gambling behavior issue by flagging behavior patterns that are associated with at-risk gambling. The idea is to give those people support resources and not label them as addicts.  

In [4]:
import pandas as pd
from poker_coach.config import BUSTABIT_DATA_FILE
from poker_coach.utils import rainbow


df = pd.read_csv(BUSTABIT_DATA_FILE)

## Step 1: Create the player_stats table

    This table is grouped by each player’s Username.
    It shows a summarization of gameplay betting behaviors. Behaviors such as average bet, maximum bet, bet volatility, total sessions, win rate, and total profit.

In [5]:
from poker_coach.config import BITS_PER_BTC, BTC_USD_RATE

df['Won'] = df['CashedOut'].notna()   # missing CashedOut = player busted (lost)
df['NetProfit'] = df['Profit'].fillna(-df['Bet'])   # busted rounds have no Profit recorded, but the full bet is lost

# Bet/Profit are recorded in bits (1 bit = 0.000001 BTC), convert to BTC so the values read as real money
df['Bet_BTC'] = df['Bet'] / BITS_PER_BTC
df['NetProfit_BTC'] = df['NetProfit'] / BITS_PER_BTC


player_betting_behavior = df.groupby('Username').agg(
    average_bet = ('Bet_BTC', 'mean'),
    max_bet = ('Bet_BTC', 'max'),
    bet_volatility = ('Bet_BTC', 'std'),
    total_sessions=('PlayDate', 'nunique'),
    win_rate=('Won', 'mean'), 
    total_profit=('NetProfit_BTC', 'sum'),
).reset_index()

rainbow(player_betting_behavior, 20, 'sample')


,Username,average_bet,max_bet,bet_volatility,total_sessions,win_rate,total_profit
1840,Wdestroier,0.000020,0.000025,0.000007,2,1.000000,0.000012
2131,berserker,0.005315,0.062633,0.013781,38,0.210526,-0.132959
2793,ipinupin,0.000457,0.001000,0.000496,11,0.454545,-0.000434
1262,Nokia_N73,0.000001,0.000001,nan,1,0.000000,-0.000001
2783,inSANE420_,0.000777,0.000777,nan,1,0.000000,-0.000777
1756,Tulus72,0.000151,0.004000,0.000507,67,0.537313,-0.000203
1302,Olexandr,0.000917,0.002700,0.000578,41,0.292683,-0.005847
1824,Vyvanse,0.003436,0.010000,0.003180,8,0.625000,-0.010886
2424,dimaklas,0.000001,0.000001,0.000000,5,0.800000,-0.000001
358,ChasingNyan,0.005918,0.025000,0.009716,9,0.333333,0.028970


Identical version of the table above, converted to USD.

In [6]:
df['Bet_USD'] = df['Bet_BTC'] * BTC_USD_RATE
df['NetProfit_USD'] = df['NetProfit_BTC'] * BTC_USD_RATE


player_betting_behavior_usd = df.groupby('Username').agg(
    average_bet = ('Bet_USD', 'mean'),
    max_bet = ('Bet_USD', 'max'),
    bet_volatility = ('Bet_USD', 'std'),
    total_sessions=('PlayDate', 'nunique'),
    win_rate=('Won', 'mean'), 
    total_profit=('NetProfit_USD', 'sum'),
).reset_index()

rainbow(player_betting_behavior_usd, 20, 'sample')


,Username,average_bet,max_bet,bet_volatility,total_sessions,win_rate,total_profit
1935,acethug37,0.007500,0.007500,0.000000,8,1.000000,0.001027
986,Krayton,0.742500,2.250000,0.826533,7,0.571429,-0.178620
253,Bill_K,0.244500,2.025000,0.522774,15,0.600000,-2.010442
3582,scbeachbum,0.262500,0.262500,nan,1,1.000000,0.123375
1842,WeekEnd,1.125000,2.250000,0.750000,4,1.000000,1.486590
682,GoHard-,8.374875,14.250000,8.308681,2,0.500000,40.281000
3103,marifemndza,1.346143,3.750000,1.213344,7,0.285714,-3.932460
1852,WinItAll,0.219750,0.219750,nan,1,0.000000,-0.219750
1796,VanillaHF,0.117350,0.124500,0.002903,15,1.000000,0.110317
1037,LiftedSpirit,153.083250,153.083250,nan,1,1.000000,56.515283
